In [1]:
import sys
sys.path.append('../src')
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement mini (from versions: none)
ERROR: No matching distribution found for mini


# Simple HandWriting OCR

This simple handwriting will involve custom CNN model by using Tensorflow.
<br>I will call it MinimNet and it can be found in model.py

## 1. Import All and Prepare the Dataset
Before going on, we could download the dataset from the [link Sueiras](https://s3-eu-west-1.amazonaws.com/handwriting-curated-database/curated.tar.gz) have given.<br>
And just unzip the file in the same directory of this notebook.

In [2]:
from model import MinimNet
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from imutils import resize, grab_contours
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from PIL import Image
import numpy as np
import cv2
import os

import all needed libraries and functions needed

And before we are going to prepare the dataset, we would want to set our parameters first

1. Dataset path,
2. Image size,
3. Batch size,
4. Number of epochs,
5. Initial Learning Rate (LR)
6. Ratio of train/test

1.1 Giải nén datasets

In [3]:
import os
import tarfile

# Khai báo tên file
output_filename = "curated.tar.gz"
input_files = ["curated.tar.gz.01", "curated.tar.gz.02"]

print(f"[*] Đang nối các file: {input_files} -> {output_filename}...")

try:
    # 1. Nối file
    with open(output_filename, 'wb') as outfile:
        for filename in input_files:
            if not os.path.exists(filename):
                print(f"[!] Lỗi: Không tìm thấy file {filename}")
                raise FileNotFoundError(filename)

            with open(filename, 'rb') as infile:
                while True:
                    chunk = infile.read(65536) # Đọc từng chunk 64KB
                    if not chunk:
                        break
                    outfile.write(chunk)
    print("[v] Nối file hoàn tất!")

    # 2. Giải nén
    print(f"[*] Đang giải nén: {output_filename} (Quá trình này có thể mất vài phút)...")
    with tarfile.open(output_filename, "r:gz") as tar:
        # Giải nén thẳng vào thư mục hiện tại, không in chi tiết (verbose) để tránh treo RAM của trình duyệt
        tar.extractall(path=".")

    print("[v] Xong! Toàn bộ file đã được giải nén.")

except Exception as e:
    print(f"[!] Có lỗi xảy ra trong quá trình xử lý: {e}")

[*] Đang nối các file: ['curated.tar.gz.01', 'curated.tar.gz.02'] -> curated.tar.gz...
[v] Nối file hoàn tất!
[*] Đang giải nén: curated.tar.gz (Quá trình này có thể mất vài phút)...


/tmp/ipykernel_8770/3426383820.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=".")


[v] Xong! Toàn bộ file đã được giải nén.


In [4]:
notebook_path = os.path.abspath('Simple-OCR.ipynb')
dataset_path = os.path.dirname(notebook_path)+'/curated'

image_size = (32, 32)
batch_size = 128
epoch = 50
init_lr = 1e-1
test_ratio = .25

After putting all the needed parameters, we then will try to load the images from the path we defined

We will need both the image and label, which the label is the folder name (ASCII Decimal)

Thus we need somekind of label to make it more uniformed.<br> Then we will make a classLabels that consist of needed images to train.

After that we will load all of the images that we need along with the labels<br>
And change the labels into a one-hot label.

In [5]:
classLabels = "1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ"
classLabels = [i for i in classLabels]

img = []
lbl = []

# Vòng lặp đi qua các thư mục trong dataset
for folder in os.listdir(dataset_path):
    # 1. BỘ LỌC THƯ MỤC: Bỏ qua các tệp/thư mục ẩn (bắt đầu bằng dấu chấm)
    if folder.startswith('.'):
        continue

    folder_path = os.path.join(dataset_path, folder)

    # Kiểm tra xem đó có phải là thư mục không
    if os.path.isdir(folder_path):
        try:
            # Chuyển tên thư mục thành số, sau đó thành ký tự ASCII
            folder_char = chr(int(folder))
        except ValueError:
            # Nếu tên thư mục không phải là số, bỏ qua
            continue

        # Nếu ký tự nằm trong danh sách nhãn cần thiết
        if folder_char in classLabels:
            lbl_idx = classLabels.index(folder_char)

            for file in os.listdir(folder_path):
                # 2. BỘ LỌC TỆP TIN: Bỏ qua các tệp ẩn bên trong thư mục (vd: ._60031.png)
                if file.startswith('.'):
                    continue

                file_path = os.path.join(folder_path, file)

                # 3. BỘ LỌC AN TOÀN: Bắt lỗi nếu có tệp bị hỏng không thể mở được
                try:
                    image = Image.open(file_path).resize(image_size)
                    img_array = np.array(image) * 1. / 255.
                    img.append(np.expand_dims(img_array, axis=-1))
                    lbl.append(lbl_idx)
                except Exception as e:
                    print(f"[Cảnh báo] Bỏ qua tệp bị lỗi: {file_path} - Chi tiết: {e}")

# Chuyển đổi list thành numpy array ở bước cuối cùng
img = np.array(img)
lbl = np.array(lbl)

print(f"Hoàn tất! Đã nạp thành công {len(img)} hình ảnh.")

Hoàn tất! Đã nạp thành công 26562 hình ảnh.


In [6]:
#create a one-hot label
le = LabelBinarizer()
lbl = le.fit_transform(lbl)

#count the total number of images per class
classTotals = lbl.sum(axis=0)
classWeight = {}

#create a custom weight for every class
for i in range(len(classTotals)):
    classWeight[i] = classTotals.max() / classTotals[i]

#split the class to train and test using scikit train_test_split
(trainImg, testImg, trainLbl, testLbl) = train_test_split(img, lbl,
                                                         test_size = test_ratio,
                                                         stratify = lbl,
                                                         random_state = 42)

After we done in splitting the dataset, we will proceed to the next step.
<br><br>

## 2. Create the Model to train

As I have said before, we will try to build a model.<br>
This model will be consisting of some layers such as:
1. Bottleneck Layer,
2. Inverted Bottleneck Layer,
3. Squeeze Global Layer,
4. SE Module.

The model could be checked in the Model.py or in the summary below.

In [9]:
model = MinimNet((image_size[0], image_size[1], 1), len(classTotals))
model.summary()

opt = SGD(learning_rate=init_lr)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=['accuracy'])

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 32, 32, 4) │        104 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 32, 32, 8) │        296 │ conv2d_32[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16, 8) │          0 │ conv2d_33[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 16, 16,    │        144 │ max_pooling2d_2[… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_56       │ (None, 16, 16,    │          0 │ conv2d_34[0][0]   │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ activation_56[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_28 │ (None, 16, 16,    │        320 │ batch_normalizat… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_57       │ (None, 16, 16,    │          0 │ depthwise_conv2d… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        128 │ activation_57[0]… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 16, 16,    │        528 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_58       │ (None, 16, 16,    │          0 │ conv2d_35[0][0]   │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ activation_58[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_29 │ (None, 16, 16,    │        160 │ batch_normalizat… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_59       │ (None, 16, 16,    │          0 │ depthwise_conv2d… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ activation_59[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_30 │ (None, 16, 16,    │        320 │ batch_normalizat… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 613,724 (2.34 MB)

 Trainable params: 604,476 (2.31 MB)

 Non-trainable params: 9,248 (36.12 KB)

Now after we have the model, it is time to start training the model.

## 3. Train the model

Now, we will make an augmentation for the training batch so that it could have more variations of the dataset.<br>
We will also make an additional module from tensorflow such as:

1. EarlyStopping : To stop the training if there's no improvement on the monitored variable,
2. ModelCheckpoint : To save the model/weight if it reached higher value on the monitored variable,
3. ReduceLROnPlateau : To reduce the LR if there's no improvement on the monitored variable.

In [10]:
aug = ImageDataGenerator(rotation_range=10, zoom_range=.05,
                        width_shift_range=.1, height_shift_range=.1,
                        shear_range=.15, horizontal_flip=False,
                        fill_mode='nearest')

eS = EarlyStopping(monitor='val_loss', patience=10, verbose=0, mode='min')
mChk = ModelCheckpoint('../models/OCR Model.h5', save_best_only=True, monitor='val_loss', mode='min')
rLR = ReduceLROnPlateau(monitor='val_loss', factor=.01, patience=10, verbose=1, min_lr=1e-6, mode='min')

In [ ]:
h = model.fit(aug.flow(trainImg, trainLbl, batch_size=batch_size),
             validation_data=(testImg, testLbl),
             steps_per_epoch=len(trainImg)//batch_size,
             epochs=epoch, callbacks=[eS, mChk, rLR],
             class_weight=classWeight, verbose=2)

Epoch 1/50


After training, we will plot the result of the training, such as the loss and accuracy.

Here, we will use the pyplot to plot the result into an image called Result.png

In [ ]:
n = np.array(range(len(h.history['loss'])))
plt.style.use('ggplot')
plt.figure()
plt.plot(n, h.history['loss'], label='Train_loss')
plt.plot(n, h.history['val_loss'], label='Val_loss')
plt.title=('Training Accuracy and Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss/Accuracy')
plt.legend(loc='lower left')
plt.savefig('Result.png')

Then we will test it, and show each class result in F1 score using sklearn classification_report

In [ ]:
predictions = model.predict(testImg, batch_size=batch_size)
print(classification_report(testLbl.argmax(axis=1),
                            predictions.argmax(axis=1), target_names=classLabels))

And its finished! But only for the model part.

Next, we will combine this with OpenCV to make it into real OCR and read multiple character in the images at once.

## 4. Combining with OpenCV
In this step we will:
1. Load an image containing handwriting using OpenCV,
2. Convert the image into Grayscale,
3. Apply some image processing,
4. Get the contours and sort them from left to right,
5. Crop images based on the contours,
6. Predict each image from the cropped image,
7. Append the result to the image.

In [ ]:
# put all the needed variable/parameter
file_name = 'image_to_test.jpg'
image_path = os.path.dirname(notebook_path)+'/'+file_name

image_to_test = cv2.imread(image_path)

#convert to grayscale
gray = cv2.cvtColor(image_to_test, cv2.COLOR_BGR2GRAY)
#blurring the image
blur = cv2.GaussianBlur(gray, (5, 5), 0)

#apply edge filter to the blurred image
edge = cv2.Canny(blur, 30, 150)

In [ ]:
#get contours
cntrs = cv2.findContours(edge.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cntrs = grab_contours(cntrs)

#sort contours from top left to bottom right
boundingBox = [cv2.boundingRect(c) for c in cntrs]
(cntrs, _) = zip(*sorted(zip(cntrs, boundingBox), key=lambda x: (-x[1][1], x[1][0])))

In [ ]:
char_detected = []

for c in cntrs:
    (x,y,w,h) = cv2.boundingRect(c)

    #attemp to ignore small contours
    if (w >= 5 and w <= 350) and (h >= 15 and h <= 320):
        #create an Region of Interest
        roi = gray[y:y+h, x:x+w]

        #threshold it to make it into binary image
        bin_img = cv2.threshold(roi, 0 ,255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
        (iH, iW) = bin_img.shape

        #if imageWidth(iW) is bigger than the Height, then we will scale the image until
        #the width is equal to 32, else scale the image until we got the height is the 32
        #32 used in here is based on the image_size which is (32,32)
        if iW > iH:
            bin_img = resize(bin_img, width = image_size[0])
        else:
            bin_img = resize(bin_img, height = image_size[1])

        #update with the new image size
        (iH, iW) = bin_img.shape
        dX = int(max(0, 32 - iW) / 2.0)
        dY = int(max(0, 32 - iH) / 2.0)

        #pad the image and resize it to match the input of the model
        padded = cv2.copyMakeBorder(bin_img, top=dY, bottom=dY,
            left=dX, right=dX, borderType=cv2.BORDER_CONSTANT,
            value=(0, 0, 0))
        padded = cv2.resize(padded, image_size)

        #normalize the image so that it got the same value of testImg
        padded = padded.astype("float32") / 255.0
        padded = np.expand_dims(padded, axis=-1)

        #put it into list of the queue of images to predict
        char_detected.append([padded, (x, y, w, h)])

#get each location and image of detected char
boxes = [b[1] for b in char_detected]
char_detected = np.array([c[0] for c in char_detected], dtype="float32")
#feed it as the input of the model
predictions = model.predict(char_detected)
preds = []
for pred in predictions:
    i = np.argmax(pred)
    prob = pred[i]
    preds.append([classLabels[i], prob])

for (pred, (x, y, w, h)) in zip(preds, boxes):
    #put the result into the image and print it
    print("[DETECTED CHAR] {} - {:.2f}%".format(pred[0], pred[1] * 100))
    cv2.rectangle(image_to_test, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(image_to_test, "{}".format(pred[0]), (x - 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

#show the result
cv2.imwrite("Image_after_test.jpg", image_to_test)

In [ ]:
#show the result
plt.imshow(image_to_test)

As we can see the result shows:
<br>
"This is only a jimple ocr 2020 thij nekds momd troaining"

There are some errors on the S and The insides of R, which always shows up as D or O.<br>
And RE that connected also read as M.

And that's it! Congratulation, you have reached the end of this tutorial!

But as we can see that the result is still not that good.<br>
There are still things that we can do to fix this such as:
1. Make the the minimum size bigger,
2. Maybe train it more or try to use other network,
3. Try to train using other dataset.